In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../..'))
import geopandas as gpd
import ee

gdf = gpd.read_file("gpkgs/bda_2024_patches_2024_classified.gpkg")[:10]




In [24]:
cloud_project = "hedgementation"

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

In [35]:
gdf2 = gpd.read_file("gpkgs/bda_2024_patches_2024_classified.gpkg")
len(gdf2)

603564

In [25]:
import datetime


nb_pixel = 128
scale = 10
selected_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

start_year = "2021"
end_year = "2022"

datetime_format = "%Y%m%d"

start_date = datetime.datetime.strptime(f"{start_year}0917", datetime_format)
end_date = datetime.datetime.strptime(f"{end_year}1027", datetime_format)

date_range = [start_date + datetime.timedelta(days=i * 5) 
              for i in range(int((end_date - start_date).days / 5))]


destination_folder = "/content/gdrive/My Drive/ExportTest"

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filter(ee.Filter.date(
    ee.Date(start_date.strftime("%Y-%m-%d")),
    ee.Date(end_date.strftime("%Y-%m-%d"))
)).select(selected_bands)

In [26]:
def shapely_to_ee(poly): 
    geo = poly.__geo_interface__
    ee_polygon = ee.Geometry(geo)
    return ee_polygon

ee_polys = gdf["geometry"].map(lambda x: shapely_to_ee(x))

In [27]:
def get_bounding_box(patch, crs=""):
    patch_bounds = patch["bounds"]
    return patch_bounds

def stack_images(img_collection):
    img_collection = img_collection.sort('system:time_start')
    images_list = img_collection.toList(img_collection.size())
    first = ee.Image(images_list.get(0))
    rest = ee.List(images_list.slice(1))

    def stack(img, previous):
        return ee.Image(previous).addBands(ee.Image(img))

    return ee.Image(rest.iterate(stack, first))

def rename_bands(img):
    date = img.date().format('YYYY-MM-DD')
    band_names = img.bandNames().map(lambda x: ee.String(date).cat('_').cat(x))
    img = img.rename(band_names)
    return img

def get_stacked_img_for_patch(geom,image_collection, crs):
   
    filtered_collection = image_collection.filterBounds(
        geom
        ).filter(
        ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)
        )

    stacked_image = stack_images(filtered_collection.map(rename_bands))
    return stacked_image





In [28]:
crs = ee.Projection(gdf.crs)
img = get_stacked_img_for_patch(ee_polys[0], s2, crs)


In [29]:
img.getInfo()

{'type': 'Image',
 'bands': [{'id': '2021-09-264_B2',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [10980, 10980],
   'crs': 'EPSG:32630',
   'crs_transform': [10, 0, 300000, 0, -10, 5400000]},
  {'id': '2021-09-264_B3',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [10980, 10980],
   'crs': 'EPSG:32630',
   'crs_transform': [10, 0, 300000, 0, -10, 5400000]},
  {'id': '2021-09-264_B4',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [10980, 10980],
   'crs': 'EPSG:32630',
   'crs_transform': [10, 0, 300000, 0, -10, 5400000]},
  {'id': '2021-09-264_B5',
   'data_type': {'type': 'PixelType',
    'precision': 'int',
    'min': 0,
    'max': 65535},
   'dimensions': [5490, 5490],
   'crs': 'EPSG:32630',
   'crs_transform': [20, 0, 300000, 0, -20, 5400000]},
  {'id': '2021-09-264_B6',
   'data_typ

In [30]:
import requests 
import io
import rasterio
import numpy as np

url = img.getDownloadURL({
        'scale': scale,
        'crs': "EPSG:3857",
        'format': 'GEO_TIFF',
        'region': ee_polys[0]
    })

response = requests.get(url)

tiff_bytes = io.BytesIO(response.content)
with rasterio.open(tiff_bytes) as infile:
    img_array = np.array(infile.read())
    img_array

In [31]:
img_array.shape

(420, 129, 129)

In [ ]:
def export_patch(id, geom, img, crs='EPSG:3857', scale=10):
    task = ee.batch.Export.image.toDrive(
            image=img.clip(geom),
            description=f'export_X_{id}',
            folder=destination_folder,
            fileNamePrefix=f'hedgementation_X_{id}',
            region=geom,
            scale=scale,
            maxPixels=1e10,
            crs=crs,
            fileFormat='GeoTIFF'
        )

    task.start()
    return task

test_geom = ee_polys[0]
export_patch(0, test_geom, get_stacked_img_for_patch(ee_polys[0], s2, crs))

<Task OJBSQTRYJFBSUIREG5F5SW3Q EXPORT_IMAGE: export_0 (UNSUBMITTED)>